# Digital Payments Process Analysis — Exploratory Data Analysis
**Analyst:** Sagar Kandelkar | **Date:** September 2026
**Data:** Synthetic digital payments dataset for portfolio case study

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
payments = pd.read_csv('../data/payments.csv')
merchants = pd.read_csv('../data/merchants.csv')
customers = pd.read_csv('../data/customers.csv')
failures = pd.read_csv('../data/payment_failures.csv')

print('Payments:', payments.shape)
print('Merchants:', merchants.shape)
print('Customers:', customers.shape)
print('Failures:', failures.shape)

## 2. Payment Success Rate by Method

In [ ]:
method_stats = payments.groupby('payment_method')['status'].value_counts().unstack(fill_value=0)
method_stats['total'] = method_stats.sum(axis=1)
method_stats['success_rate'] = (method_stats['success'] / method_stats['total']) * 100

method_stats[['success', 'failed']].plot(kind='bar', stacked=True, color=['#2563eb', '#dc2626'])
plt.title('Payment Volume by Method & Status')
plt.xlabel('Payment Method')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(['Success', 'Failed'])
plt.tight_layout()
plt.show()

print(method_stats[['success', 'failed', 'success_rate']])

## 3. Daily Payment Volume & Failure Rate

In [ ]:
payments['payment_date'] = pd.to_datetime(payments['payment_date'])
daily = payments.groupby('payment_date').agg(
    total=('payment_id', 'count'),
    success=('status', lambda x: (x == 'success').sum()),
    failed=('status', lambda x: (x == 'failed').sum())
).reset_index()
daily['failure_rate'] = (daily['failed'] / daily['total']) * 100

fig, ax1 = plt.subplots()
ax1.bar(daily['payment_date'].astype(str), daily['total'], color='#2563eb', alpha=0.7, label='Total')
ax1.set_xlabel('Date')
ax1.set_ylabel('Transaction Count', color='#2563eb')
ax1.tick_params(axis='y', labelcolor='#2563eb')
ax1.set_xticklabels(daily['payment_date'].astype(str), rotation=45)

ax2 = ax1.twinx()
ax2.plot(daily['payment_date'].astype(str), daily['failure_rate'], color='#dc2626', marker='o', linewidth=2, label='Failure %')
ax2.set_ylabel('Failure Rate %', color='#dc2626')
ax2.tick_params(axis='y', labelcolor='#dc2626')

plt.title('Daily Volume vs Failure Rate')
plt.tight_layout()
plt.show()

## 4. Failure Reason Breakdown

In [ ]:
failure_reasons = payments[payments['status'] == 'failed']['failure_reason'].value_counts()
failure_reasons.plot(kind='barh', color='#dc2626')
plt.title('Failure Reasons')
plt.xlabel('Count')
plt.tight_layout()
plt.show()
print(failure_reasons)

## 5. Merchant Category Volume

In [ ]:
merchant_vol = payments.merge(merchants, on='merchant_id').groupby('merchant_category')['amount'].sum().sort_values(ascending=False)
merchant_vol.plot(kind='bar', color='#2563eb')
plt.title('Transaction Volume by Merchant Category')
plt.xlabel('Category')
plt.ylabel('Total INR')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Settlement Analysis

In [ ]:
settled = payments[(payments['status'] == 'success') & (payments['settlement_status'] == 'settled')].copy()
settled['settlement_date'] = pd.to_datetime(settled['settlement_date'])
settled['settlement_days'] = (settled['settlement_date'] - settled['payment_date']).dt.days

sns.boxplot(x='payment_method', y='settlement_days', data=settled, palette='Blues')
plt.title('Settlement Days by Payment Method')
plt.xlabel('Payment Method')
plt.ylabel('Days to Settle')
plt.tight_layout()
plt.show()

print(settled.groupby('payment_method')['settlement_days'].agg(['mean', 'median', 'max']))

## 7. Customer LTV vs Payment Preference

In [ ]:
ltv_pref = customers.groupby(['preferred_payment_method', 'lifetime_value_segment']).size().unstack(fill_value=0)
ltv_pref.plot(kind='bar', stacked=True, color=['#94a3b8', '#2563eb', '#0f172a'])
plt.title('Customer LTV Segment by Payment Preference')
plt.xlabel('Payment Method')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='LTV Segment')
plt.tight_layout()
plt.show()

## 8. Key Insights & Recommendations

1. **Success Rate:** Overall success rate is ~80% in this sample — focus needed on UPI and Card channels.
2. **Failure Patterns:** Network timeouts and insufficient funds are top failure reasons — retry logic and balance checks can help.
3. **Merchant Categories:** E-commerce and Travel drive highest volumes — ensure dedicated capacity.
4. **Settlement:** Most settlements complete within T+1; NetBanking shows slightly longer cycles.
5. **Customer Segments:** UPI preferred by high-LTV customers — invest in UPI reliability.
6. **Unresolved Failures:** VPA not found and card expired issues remain unresolved — proactive customer communication needed.

### Recommendations
- Implement pre-transaction balance validation for wallets and bank accounts
- Deploy smart retry with exponential backoff for network failures
- Proactively notify customers of expired cards 30 days before expiry
- Add VPA validation at payment initiation to reduce 'not found' errors
- Prioritize E-commerce and Travel merchant gateway capacity during peak hours